# U-Net land-cover training -- Dynamic World labels trial (Colab)

**checkpoint_DynamicWorld branch.** Same architecture and training loop as
[`train_unet.ipynb`](train_unet.ipynb), with only the training LABEL SOURCE
swapped from ESA WorldCover to Dynamic World -- see
[`docs/dynamic_world_vs_worldcover_2026-09-27.md`](../../docs/dynamic_world_vs_worldcover_2026-09-27.md)
for why, and the RF-only gate result this follows on from
(`scripts/train_landcover_rf_dynamicworld.py`: accuracy 65.6% -> 72.6% against
the same 300 hand-labeled points).

**Every output of this run is kept separate from the production model** --
its own local path (`models/unet_landcover_dw_trial.keras`), its own GCS
prefix, its own GEE training-patch cache key. Running this notebook cannot
overwrite `models/unet_landcover.keras` or anything `pull_models.py --model
unet` reads. Run this from **VS Code**: `Select Kernel` -> `Colab` -> sign in
-> `New Colab Server` -> pick **GPU** (the free T4 is enough) -> connect.

Local machines never train this model -- only run inference on the weights
this notebook produces. After this notebook finishes, on your own machine run:

```bash
python scripts/run_landcover_unet_inference_dynamicworld_trial.py --with-probabilities
```

That single command downloads the trial weights (verified via sha256,
mirroring `pull_models.py`'s own check) and runs full-Singapore CPU inference
-- no separate pull step, since this model isn't in `pull_models.py`'s
`--model` list by design (keeping the two label sources' weights unmistakably
separate).

**One-time setup**: none needed in advance -- the auth cell below prompts a
masked paste-in box for this repo's service-account JSON key
(`credentials/nus-iss-urban-heat-sg-*.json`) each session, via `getpass` (a
plain Jupyter stdin prompt). Never an interactive Google login, matching this
project's service-account-only convention everywhere else. (`google.colab.
files.upload()` hung indefinitely through VS Code's remote-kernel connection
to Colab, confirmed 2026-08-05 -- see `train_unet.ipynb`'s own auth cell.)


In [1]:
# --- Repo sync -----------------------------------------------------------
# Colab's /content disk is empty every session -- clone (or pull, if this
# session already has it) the public repo so `src/`/`config/` are importable
# the same "sys.path manipulation, no packaging" way every local script uses.
import os
import subprocess
import sys

REPO_URL = "https://github.com/EngineerKX/urban-heat-cooling-priority.git"
REPO_DIR = "/content/urban-heat-cooling-priority"
# checkpoint_DynamicWorld, NOT kxbranch -- this branch carries
# src/ingest/dynamic_world.py, rf_baseline.py's class_band/asset_id params,
# and unet_data.py's class_band/label_source-salted-fingerprint params that
# this notebook's cells below depend on. Must already be PUSHED to the
# remote (commit + push your local changes on checkpoint_DynamicWorld before
# running this cell) -- a purely local commit is invisible here.
REPO_BRANCH = "checkpoint_DynamicWorld"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True
    )
else:
    # Existing session: make sure we're on REPO_BRANCH and up to date, not
    # still on whatever was cloned earlier.
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(
        ["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR, check=True
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repo ready at", REPO_DIR, "on branch", REPO_BRANCH)


Repo ready at /content/urban-heat-cooling-priority on branch checkpoint_DynamicWorld


In [2]:
# --- Install trimmed training-only dependencies ---------------------------
# requirements-colab.txt deliberately excludes tensorflow (Colab's
# pre-installed, GPU-matched build).
%pip install -q -r {REPO_DIR}/requirements-colab.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# --- Auth: service account, not interactive Google login -------------------
# Paste the full contents of your service-account JSON key when prompted
# below (open credentials/nus-iss-urban-heat-sg-*.json in a text editor,
# copy everything, Ctrl+V into the box that appears). Uses getpass -- a
# plain stdin prompt, standard Jupyter protocol -- not a browser widget,
# since google.colab.files.upload() hung indefinitely through VS Code's
# remote-kernel connection to Colab (confirmed 2026-08-05). getpass also
# never echoes the input back into the cell's saved output, unlike a plain
# input() -- this repo is public on GitHub, so that distinction matters.
import getpass
import os

key_json = getpass.getpass("Paste the full contents of your service-account JSON key, then press Enter: ")

key_path = "/content/gee_key.json"
with open(key_path, "w") as f:
    f.write(key_json)

os.environ["GEE_PRIVATE_KEY_PATH"] = key_path
os.environ["GEE_SERVICE_ACCOUNT"] = "urban-heat-pipeline@nus-iss-urban-heat-sg.iam.gserviceaccount.com"
os.environ["GEE_PROJECT_ID"] = "nus-iss-urban-heat-sg"
os.environ["GEE_EXPORT_BUCKET"] = "nus-iss-urban-heat-sg-exports"
print("Service-account credentials staged.")

Service-account credentials staged.


In [4]:
import ee
import mlflow
import pandas as pd
import tensorflow as tf

from config import settings
from config.settings import (
    DRY_SEASON_MONTHS,
    DW_TRAIN_END,
    DW_TRAIN_START,
    GCS_MODEL_BUCKET,
    S2_CLOUD_PROB_MAX,
    S2_UTM_CRS,
    SG_BBOX,
    TARGET_SCALE_M,
    VALIDATION_SAMPLE_GCS_PREFIX,
    YEARS,
)
from src.ingest.dynamic_world import get_dynamicworld_bucket_image
from src.ingest.gee import init_ee
from src.ingest.subzones import as_ee_feature_collection, dissolve_boundary, fetch_subzones_geojson
from src.landcover.rf_baseline import build_feature_image, build_training_region
from src.landcover.unet_data import (
    build_training_stack,
    export_training_patches,
    parse_training_patches,
    training_fingerprint,
)
from src.landcover.unet_train import train_unet
from src.utils import gcs
from src.utils.experiment_tracking import export_run_summary, start_run
from src.utils.seed import set_all_seeds
from pathlib import Path

# This trial's own paths/prefixes -- distinct from UNET_MODEL_SAVE_PATH /
# UNET_MODEL_GCS_PREFIX (config/settings.py) everywhere, so nothing here can
# collide with the production WorldCover-trained U-Net.
UNET_MODEL_SAVE_PATH_TRIAL = Path("models/unet_landcover_dw_trial.keras")
UNET_MODEL_GCS_PREFIX_TRIAL = "models/unet_landcover_dw_trial"


In [5]:
set_all_seeds()
init_ee()
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", bool(gpus), "-", gpus[0].name if gpus else "none")

EE initialized OK (service account), project: nus-iss-urban-heat-sg
GPU available: False - none


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [6]:
# --- Preview the model architecture (no training, no GPU needed) ----------
# Every Conv2D / MaxPooling2D / Conv2DTranspose / Concatenate layer, its
# output shape, and parameter count -- printed here so it's visible
# regardless of whether the training cell below ends up actually training
# (a cache hit skips straight past model.summary() inside train_unet()).
# Handy for tuning UNET_BASE_FILTERS / UNET_PATCH_SIZE in
# config/settings.py before committing to a full training run: just
# re-run this cell after editing settings to see the new shapes/param
# counts immediately. Builds the graph only -- not compiled or trained.
from config.settings import ALL_FEATURE_BANDS, UNET_BASE_FILTERS, UNET_PATCH_SIZE
from src.landcover.unet_data import N_CLASSES
from src.landcover.unet_model import build_unet

preview_model = build_unet(
    (UNET_PATCH_SIZE, UNET_PATCH_SIZE, len(ALL_FEATURE_BANDS)),
    n_classes=N_CLASSES, base_filters=UNET_BASE_FILTERS,
)
preview_model.summary()

Model: "unet_landcover"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 9)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │      2,624 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │     36,928 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 16, 16,    │    590,080 │ conv2d_6[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 32, 32,    │    131,200 │ conv2d_7[0][0]    │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 32, 32,    │    295,040 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 32, 32,    │    147,584 │ conv2d_8[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 64, 64,    │     32,832 │ conv2d_9[0][0]  

 Total params: 1,927,428 (7.35 MB)

 Trainable params: 1,927,428 (7.35 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# --- Pull the hand-labeled validation sample -------------------------------
# Gitignored (data/interim/**), so a fresh git clone doesn't bring it along.
# It can't be regenerated in Colab either -- labeling requires a human and
# the Streamlit app. Pushed once (and after every relabel) via:
#   python -c "from src.utils import gcs; from config.settings import GCS_MODEL_BUCKET, VALIDATION_SAMPLE_GCS_PREFIX; gcs.upload_file('data/interim/validation_sample/validation_sample_300_labeled.csv', GCS_MODEL_BUCKET, f'{VALIDATION_SAMPLE_GCS_PREFIX}.csv')"
from pathlib import Path

VALIDATION_CSV = Path("data/interim/validation_sample/validation_sample_300_labeled.csv")
VALIDATION_CSV.parent.mkdir(parents=True, exist_ok=True)
gcs.download_blob(GCS_MODEL_BUCKET, f"{VALIDATION_SAMPLE_GCS_PREFIX}.csv", VALIDATION_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)
print(f"Loaded {len(validation_df)} validation points.")

Loaded 300 validation points.


In [8]:
# --- Build the GEE feature stack -------------------------------------------
# Identical to train_unet.ipynb's own cell EXCEPT the label source: Dynamic
# World instead of WorldCover. Same feature bands/season window/exclusion
# buffer as the RF baseline (and the WorldCover-trained U-Net), kept
# deliberately identical so this comparison isolates the label source, same
# reasoning as scripts/train_landcover_rf_dynamicworld.py.
sg_bbox = ee.Geometry.Rectangle(list(SG_BBOX))
subzones_fc = as_ee_feature_collection(fetch_subzones_geojson())
boundary = dissolve_boundary(subzones_fc)

feature_image, valid_mask = build_feature_image(sg_bbox, boundary, YEARS, DRY_SEASON_MONTHS, S2_CLOUD_PROB_MAX)
dw_bucket_image = get_dynamicworld_bucket_image(
    boundary, S2_UTM_CRS, TARGET_SCALE_M, DW_TRAIN_START, DW_TRAIN_END, valid_mask=valid_mask,
)

# Folded into the training-patch cache key below (training_fingerprint's
# label_source) so this run can never silently reuse -- or be reused by --
# the WorldCover-trained U-Net's cached GCS patches, even though both share
# the exact same validation CSV.
LABEL_SOURCE = f"dynamicworld_{DW_TRAIN_START}_{DW_TRAIN_END}"

training_region, _ = build_training_region(boundary, validation_df)
training_stack = build_training_stack(feature_image, dw_bucket_image, training_region, class_band="dw_class")
print("Feature stack built (Dynamic World labels).")


Dissolved Singapore boundary built. Approx area: 788.3 km²
(Sanity check: Singapore's land area is ~730-735 km².)
Dynamic World images 2021-01-01 to 2027-01-01: 88
Singapore boundary area: 788.30 km²
Training region area (post-exclusion): 788.09 km²
Excluded around validation points: 208,077 m²
✅ Validation points are spatially excluded from the training region.
Feature stack built (Dynamic World labels).


In [9]:
# --- Export (or reuse the GCS-cached) training patches ---------------------
# export_training_patches checks gs://<bucket>/unet_train_patches/<fingerprint>/
# first -- only the very first run for this label source actually pays for
# the multi-minute-plus GEE export; a later re-run just downloads the cached
# shards. `label_source=LABEL_SOURCE` keys this run's cache separately from
# the WorldCover-trained U-Net's (see cell above). Pass force_export=True to
# bypass deliberately.
#
# class_band="dw_class" here MUST match get_dynamicworld_bucket_image's own
# output band name -- build_training_stack exported the label under that
# literal name (not renamed to "wc_class"), so parse_training_patches has to
# be told the same name to read it back. A mismatch here is exactly what
# failed the first run of this notebook (InvalidArgumentError: Feature:
# wc_class ... could not be found) -- src/landcover/unet_data.py fixed to
# take this as an explicit parameter instead of hardcoding it.
train_patch_dir = export_training_patches(
    training_stack, training_region, VALIDATION_CSV, label_source=LABEL_SOURCE,
)
train_ds, val_ds, n_patches = parse_training_patches(train_patch_dir, class_band="dw_class")


Training patches at C:\content\urban-heat-cooling-priority\data\interim\unet_patches\train match the current validation CSV — skipping export (pass force=True to redo anyway).
Usable patches (after dropping all-nodata ones): 631
Train patches: 536, validation patches: 95


In [10]:
# --- Train -------------------------------------------------------------
# force_retrain=True: always retrains from scratch, even if a cached model
# from an earlier run in this same Colab session is already sitting on
# /content disk. Set back to False if you just want to re-run this cell
# safely while debugging downstream code without paying for a full retrain.
data_fingerprint = training_fingerprint(VALIDATION_CSV, LABEL_SOURCE)

with start_run("unet_dynamicworld_trial"):
    mlflow.log_params({
        "unet_patch_size": settings.UNET_PATCH_SIZE,
        "unet_batch_size": settings.UNET_BATCH_SIZE,
        "unet_epochs": settings.UNET_EPOCHS,
        "unet_learning_rate": settings.UNET_LEARNING_RATE,
        "unet_base_filters": settings.UNET_BASE_FILTERS,
        "unet_early_stop_patience": settings.UNET_EARLY_STOP_PATIENCE,
        "unet_train_val_split": settings.UNET_TRAIN_VAL_SPLIT,
        "label_source": LABEL_SOURCE,
        "n_validation_points": len(validation_df),
        "n_train_patches": n_patches,
        "force_retrain": True,
    })

    model, history = train_unet(
        train_ds, val_ds, model_save_path=UNET_MODEL_SAVE_PATH_TRIAL,
        data_fingerprint=data_fingerprint, force_retrain=True,
    )

    if history is not None:
        for epoch, (loss, train_metric, val_loss, val_metric) in enumerate(
            zip(history["train_loss"], history["train_metric"], history["val_loss"], history["val_metric"])
        ):
            mlflow.log_metrics(
                {"train_loss": loss, "train_accuracy": train_metric, "val_loss": val_loss, "val_accuracy": val_metric},
                step=epoch,
            )
        mlflow.log_metric("best_val_loss", min(history["val_loss"]))

print(f"Model saved to {UNET_MODEL_SAVE_PATH_TRIAL}")


2026/09/28 07:15:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/28 07:15:12 INFO mlflow.store.db.utils: Updating database tables
2026/09/28 07:15:13 INFO mlflow.tracking.fluent: Experiment with name 'landcover_classifiers' does not exist. Creating a new experiment.


Model: "unet_landcover"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 9)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 128, 128,  │      2,624 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 128, 128,  │      9,248 │ conv2d_15[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ conv2d_16[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 64, 64,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 64, 64,    │     36,928 │ conv2d_17[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 32, 32,    │          0 │ conv2d_18[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 32, 32,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 32, 32,    │    147,584 │ conv2d_19[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 16, 16,    │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 16, 16,    │    295,168 │ max_pooling2d_5[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 16, 16,    │    590,080 │ conv2d_21[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_3  │ (None, 32, 32,    │    131,200 │ conv2d_22[0][0]   │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 32, 32,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_20[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │    295,040 │ concatenate_3[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │    147,584 │ conv2d_23[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_4  │ (None, 64, 64,    │     32,832 │ conv2d_24[0][0] 

 Total params: 1,927,428 (7.35 MB)

 Trainable params: 1,927,428 (7.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30


c:\Users\kxloo\OneDrive\Documents\VS Workspace\urban-heat-cooling-priority\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


67/67 ━━━━━━━━━━━━━━━━━━━━ 21s 270ms/step - accuracy: 0.7143 - loss: 8.3023 - val_accuracy: 0.7757 - val_loss: 1.1881
Epoch 2/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 18s 274ms/step - accuracy: 0.7789 - loss: 1.1235 - val_accuracy: 0.8297 - val_loss: 0.5727
Epoch 3/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 19s 288ms/step - accuracy: 0.8231 - loss: 0.5794 - val_accuracy: 0.8609 - val_loss: 0.3778
Epoch 4/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 20s 292ms/step - accuracy: 0.8518 - loss: 0.4016 - val_accuracy: 0.8202 - val_loss: 0.3823
Epoch 5/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 19s 291ms/step - accuracy: 0.8616 - loss: 0.3389 - val_accuracy: 0.8761 - val_loss: 0.2597
Epoch 6/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 20s 306ms/step - accuracy: 0.8786 - loss: 0.2813 - val_accuracy: 0.8893 - val_loss: 0.2209
Epoch 7/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 34s 516ms/step - accuracy: 0.8729 - loss: 0.2829 - val_accuracy: 0.8782 - val_loss: 0.2364
Epoch 8/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 20s 292ms/step - accuracy: 0.8824 - loss: 0.2528 - val_accuracy: 0.900

In [11]:
# --- Push the trained weights + this run's MLflow summary to GCS ----------
# Pushed directly (not via scripts/push_models.py, which only knows the
# production `--model unet` prefix) to UNET_MODEL_GCS_PREFIX_TRIAL, mirroring
# push_models.py's own sha256-sidecar convention so
# run_landcover_unet_inference_dynamicworld_trial.py can verify the download
# the same way pull_models.py does for the production model.
import hashlib
import json
import time


def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


if history is not None:
    local_hash = _sha256_file(UNET_MODEL_SAVE_PATH_TRIAL)
    gcs.upload_file(UNET_MODEL_SAVE_PATH_TRIAL, GCS_MODEL_BUCKET, f"{UNET_MODEL_GCS_PREFIX_TRIAL}.keras")
    gcs.upload_text(local_hash, GCS_MODEL_BUCKET, f"{UNET_MODEL_GCS_PREFIX_TRIAL}.sha256")
    print(f"Pushed trial weights -> gs://{GCS_MODEL_BUCKET}/{UNET_MODEL_GCS_PREFIX_TRIAL}.keras")

    summary = export_run_summary(
        "unet_dynamicworld_trial", "landcover_classifiers",
        params={
            "unet_patch_size": settings.UNET_PATCH_SIZE,
            "unet_batch_size": settings.UNET_BATCH_SIZE,
            "unet_epochs": settings.UNET_EPOCHS,
            "unet_learning_rate": settings.UNET_LEARNING_RATE,
            "unet_base_filters": settings.UNET_BASE_FILTERS,
            "unet_early_stop_patience": settings.UNET_EARLY_STOP_PATIENCE,
            "unet_train_val_split": settings.UNET_TRAIN_VAL_SPLIT,
            "label_source": LABEL_SOURCE,
            "n_validation_points": len(validation_df),
            "n_train_patches": n_patches,
            "force_retrain": True,
        },
        metrics_history={
            "train_loss": history["train_loss"],
            "train_accuracy": history["train_metric"],
            "val_loss": history["val_loss"],
            "val_accuracy": history["val_metric"],
        },
        final_metrics={"best_val_loss": min(history["val_loss"])},
    )
    gcs.upload_text(json.dumps(summary), GCS_MODEL_BUCKET, f"training_runs/unet_dynamicworld_trial_{int(time.time())}.json")
    print("Run summary pushed for local MLflow import (python scripts/pull_models.py imports it).")


Pushed trial weights -> gs://nus-iss-urban-heat-sg-exports/models/unet_landcover_dw_trial.keras
Run summary pushed for local MLflow import (python scripts/pull_models.py imports it).


## Next steps (on your own machine, no GPU needed)

```bash
python scripts/run_landcover_unet_inference_dynamicworld_trial.py --with-probabilities
```

This one command downloads the trial weights from
`gs://<bucket>/models/unet_landcover_dw_trial.keras` (verified via sha256,
same check `pull_models.py` does), then runs full-Singapore CPU inference +
tile reconstruction -- U-Net inference was never the GPU bottleneck, only the
training loop above was. It writes `data/processed/landcover/
unet_landcover_dw_trial.tif` (+ a probability raster) and scores it against
the same 300 hand-labeled points, alongside the existing RF/U-Net/hybrid
numbers -- the production `unet_landcover.tif`, `models/unet_landcover.keras`,
and `comparison_table.csv` are all untouched. From there, the hybrid rebuild
(RF-DW + U-Net-DW) and the merge decision follow the plan already agreed on
`checkpoint_DynamicWorld`.
